# Creating NWB Files with pynwb

[Neurodata Without Borders (NWB)](https://www.nwb.org/) is a community standard for sharing neurophysiology data. An NWB file is, on disk, an HDF5 file with a fixed schema. Below is a non-exhaustive list of the groups within an NWB file:

-  `/general` for metadata
-  `/acquisition` for raw recordings
-  `/processing` for processed data
-  `/analysis` for results of analysis
-  `/stimuli` for stimuli used in experiment
-  `/intervals/trials` for trial information
-  `/units` for sorted units

In this session you'll build a small NWB file from scratch using [pynwb](https://pynwb.readthedocs.io/) — adding the experiment and subject metadata, an extracellular signal with its electrode table, a trial table, a units table, and a couple of processing modules. At the end you'll write the file to disk and read it back to confirm everything.

## Setup

### Import Libraries

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime, timezone
from uuid import uuid4

import pynwb
from pynwb import NWBFile, NWBHDF5IO, TimeSeries
from pynwb.file import Subject
from pynwb.ecephys import ElectricalSeries, LFP
from pynwb.behavior import BehavioralTimeSeries

### Utility Functions

In [ ]:
class utils:
    @staticmethod
    def create_electrode_table_region(nwb: pynwb.NWBFile, probe_name: str, shank_nr: int, area: str, num_electrodes: int):

        device = nwb.create_device(name=probe_name, description="Synthetic 64-channel probe")

        group = nwb.create_electrode_group(
            name=f"shank{shank_nr}", description=f"single shank in {area}", location=area, device=device,
        )

        if not nwb.electrodes:
            for i in range(num_electrodes):
                nwb.add_electrode(x=0.0, y=0.0, z=float(i),location=area, group=group, filtering="none",)

            region = nwb.create_electrode_table_region(region=list(range(num_electrodes)), description=f"all {area} electrodes",)
        else:
            num_prior_electrodes = len(nwb.electrodes)
            for i in range(num_prior_electrodes, num_prior_electrodes + num_electrodes, 1):
                nwb.add_electrode(x=0.0, y=0.0, z=float(i),location=area, group=group, filtering="none",)

            region = nwb.create_electrode_table_region(region=list(range(num_prior_electrodes, num_prior_electrodes + num_electrodes, 1)), description=f"all {area} electrodes",)

        return region

## Creating an NWBFile and Subject Metadata

### Background

Every NWB file starts from an `NWBFile` object. Three fields are required:

- `session_description`: free-text description of the recording session
- `identifier`: a globally unique string (a UUID is a safe default)
- `session_start_time`: a timezone-aware `datetime` for t=0 of the recording

Optional fields can be added to capture the rest of the experimental context, such as: `experimenter`, `lab`, `institution`, `session_id`, `experiment_description`, and a `Subject` object describing the recorded animal.

### Exercises

| Code | Description |
| --- | --- |
| `NWBFile(session_description=, identifier=, session_start_time=)` | Construct an `NWBFile` |
| `str(uuid4())` | Generate a unique identifier |
| `datetime(2026, 5, 1, 10, 0, tzinfo=timezone.utc)` | Build a timezone-aware datetime |
| `nwb.subject = Subject(subject_id=, species=, sex=, age=)` | Construct a subject and attach it to the file|
| `nwb.experimenter = 'Dr. X'` | Set the experimenter (an optional metadata field) |
| `nwb.lab = 'My Lab'` | Set the lab |
| `nwb.institution = 'My University'` | Set the institution |
| `nwb.session_id = 'vis-001'` | Set a session identifier |
| `nwb.experiment_description = '...'` | Set a one-sentence experiment description |
| `with NWBHDF5IO("my_session.nwb", "w") as io:`<br>&nbsp;&nbsp;&nbsp;&nbsp;`io.write(nwb)`| Write to file |  
| `with NWBHDF5IO("my_session.nwb", "r") as io:`<br>&nbsp;&nbsp;&nbsp;&nbsp;`rnwb = io.read()` | Read the file |


**Example**: create an `NWBFile` with a description, a unique identifier, and a start time.


In [ ]:
nwb_vis = NWBFile(
    session_description="Visual stimulation in V1",
    identifier=str(uuid4()),
    session_start_time=datetime(2026, 5, 1, 10, 0, tzinfo=timezone.utc),
)
nwb_vis

root pynwb.file.NWBFile at 0x132228272072272
Fields:
  file_create_date: [datetime.datetime(2026, 6, 22, 12, 39, 18, 945783, tzinfo=tzlocal())]
  identifier: da2e64ab-53cd-481b-bd06-e338927f9a25
  session_description: Visual stimulation in V1
  session_start_time: 2026-05-01 10:00:00+00:00
  timestamps_reference_time: 2026-05-01 10:00:00+00:00

**Exercise**: create a second `NWBFile` (call it `nwb_aud`) for a different session — for example, "Auditory stimulation in A1".


In [ ]:
nwb_aud = NWBFile(
    session_description="Auditory stimulation in A1",
    identifier=str(uuid4()),
    session_start_time=datetime(2026, 5, 2, 14, 30, tzinfo=timezone.utc),
)
nwb_aud

root pynwb.file.NWBFile at 0x132228277681712
Fields:
  file_create_date: [datetime.datetime(2026, 6, 22, 12, 39, 20, 509589, tzinfo=tzlocal())]
  identifier: 2851c9d3-71ac-463e-989d-b3968a6d41e2
  session_description: Auditory stimulation in A1
  session_start_time: 2026-05-02 14:30:00+00:00
  timestamps_reference_time: 2026-05-02 14:30:00+00:00

**Exercise**: build a `Subject` for a male mouse aged 60 days and attach it to `nwb_vis`. You can fill the Subject fields as you like, but it is recommended to follow a convention, such as:
- age: ISO 8601 Duration format, e.g., "P90D" for 90 days old
- species: The formal Latin binomial nomenclature, e.g., "Mus musculus", "Homo sapiens"
- sex: Single letter abbreviation, e.g., "F" (female), "M" (male), "U" (unknown), and "O" (other)


In [ ]:
nwb_vis.subject = Subject(
    subject_id="mouse-42",
    species="Mus musculus",
    sex="M",
    age="P60D",
)
nwb_vis.subject

subject pynwb.file.Subject at 0x131551682024784
Fields:
  age: P60D
  age__reference: birth
  sex: M
  species: Mus musculus
  subject_id: mouse-42

Run the cell below to display the `nwb_vis` object and click on "subject" to check that it was added.


In [ ]:
nwb_vis

root pynwb.file.NWBFile at 0x131551682025120
Fields:
  experimenter: Otto Normal
  file_create_date: [datetime.datetime(2026, 6, 8, 10, 42, 26, 236857, tzinfo=tzlocal())]
  identifier: 228981b1-0dc9-4d3e-b859-bd295904ce49
  institution: Demo University
  lab: Anda Demo Lab
  session_description: Visual stimulation in V1
  session_start_time: 2026-05-01 10:00:00+00:00
  subject: subject pynwb.file.Subject at 0x131551682024784
Fields:
  age: P60D
  age__reference: birth
  sex: M
  species: Mus musculus
  subject_id: mouse-42

  timestamps_reference_time: 2026-05-01 10:00:00+00:00

**Exercise**: set `experimenter`, `lab`, and `institution` on `nwb_vis` to identify who recorded the session.


In [ ]:
nwb_vis.experimenter = "Otto Normal"
nwb_vis.lab = "Anda Demo Lab"
nwb_vis.institution = "Demo University"
(nwb_vis.experimenter, nwb_vis.lab, nwb_vis.institution")

('Otto Normal', 'Anda Demo Lab', 'Demo University')

Run the cell below to display the `nwb_vis` object and see that the experimenter, lab, and institution metadata was added.


In [ ]:
nwb_vis

root pynwb.file.NWBFile at 0x131551682025120
Fields:
  experimenter: Otto Normal
  file_create_date: [datetime.datetime(2026, 6, 8, 10, 42, 26, 236857, tzinfo=tzlocal())]
  identifier: 228981b1-0dc9-4d3e-b859-bd295904ce49
  institution: Demo University
  lab: Anda Demo Lab
  session_description: Visual stimulation in V1
  session_start_time: 2026-05-01 10:00:00+00:00
  subject: subject pynwb.file.Subject at 0x131551682024784
Fields:
  age: P60D
  age__reference: birth
  sex: M
  species: Mus musculus
  subject_id: mouse-42

  timestamps_reference_time: 2026-05-01 10:00:00+00:00

**Exercise**: print `nwb_vis` and look at the output to confirm the metadata you set above is attached to the file.


In [ ]:
print(nwb_vis)

root pynwb.file.NWBFile at 0x136269432562800
Fields:
  experimenter: Otto Normal
  file_create_date: [datetime.datetime(2026, 6, 2, 12, 44, 48, 448219, tzinfo=tzlocal())]
  identifier: 998bafad-d6bb-4229-a484-6202b2782e1d
  institution: Demo University
  lab: Anda Demo Lab
  session_description: Visual stimulation in V1
  session_start_time: 2026-05-01 10:00:00+00:00
  subject: subject pynwb.file.Subject at 0x136269432563072
Fields:
  age: P60D
  age__reference: birth
  sex: M
  species: Mus musculus
  subject_id: mouse-42

  timestamps_reference_time: 2026-05-01 10:00:00+00:00



**Exercise**: write the `nwb_vis` file to `vis_session.nwb` using `NWBHDF5IO`.


In [ ]:
with NWBHDF5IO("vis_session.nwb", "w") as io:
    io.write(nwb_vis)
print("wrote vis_session.nwb")

wrote vis_session.nwb


**Exercise**: write the `nwb_aud` file to `aud_session.nwb` using `NWBHDF5IO`.


In [ ]:
with NWBHDF5IO("aud_session.nwb", "w") as io:
    io.write(nwb_aud)
print("wrote aud_session.nwb")

wrote aud_session.nwb


## Adding Probes

### Background

Before extracellular recordings can be stored, NWB needs to know *where* the signals came from. This is described by a small hierarchy of objects:

- a **device**: the physical probe or acquisition system
- an **electrode group**: a set of electrodes that belong together (e.g. one shank), attached to a device and given a brain `location`
- the **electrode table**: one row per electrode, recording its position and which electrode group the electrode belongs to
- an **electrode table region**: a reference to a subset of rows in the electrode table, used later to tell a signal which electrodes it was recorded from

### Exercises

| Code | Description |
| --- | --- |
| `nwb.create_device(name="my-device-name", description="my description")` | Register a recording device |
| `group_x = nwb.create_electrode_group(name=, description=, location=, device=)` | Create a group of electrodes attached to a device and assign it to `group_x` |
| `nwb.add_electrode(x=, y=, z=, location=, group=, filtering=)` | Add a row to the electrode table |
| `nwb.electrodes.to_dataframe()` | Inspect the electrode table |
| `nwb.create_electrode_table_region(region=, description=)` | Reference a subset of electrodes |


Run the cell below to create a new nwb object to be used in this section.

In [ ]:
nwb = NWBFile(
    session_description="Demo NWB file",
    identifier=str(uuid4()),
    session_start_time=datetime(2026, 5, 1, 10, 0, tzinfo=timezone.utc),
)
nwb

root pynwb.file.NWBFile at 0x132228272070144
Fields:
  file_create_date: [datetime.datetime(2026, 6, 22, 12, 36, 59, 814061, tzinfo=tzlocal())]
  identifier: 5647af99-4649-4609-b562-d2c0962207d1
  session_description: Demo NWB file
  session_start_time: 2026-05-01 10:00:00+00:00
  timestamps_reference_time: 2026-05-01 10:00:00+00:00

**Example**: create a recording device on the file.


In [ ]:
device_a = nwb.create_device(name="probe-A", description="Synthetic 32-channel probe")
device_a

probe-A pynwb.device.Device at 0x132228270792480
Fields:
  description: Synthetic 32-channel probe

Run the cell below to display the `nwb` object and click on "devices" to see the probe that was just added.


In [ ]:
nwb

root pynwb.file.NWBFile at 0x132228272070144
Fields:
  devices: {
    probe-A <class 'pynwb.device.Device'>
  }
  file_create_date: [datetime.datetime(2026, 6, 22, 12, 36, 59, 814061, tzinfo=tzlocal())]
  identifier: 5647af99-4649-4609-b562-d2c0962207d1
  session_description: Demo NWB file
  session_start_time: 2026-05-01 10:00:00+00:00
  timestamps_reference_time: 2026-05-01 10:00:00+00:00

**Exercise**: create a second device named `'probe-B'` with a short description.


In [ ]:
device_b = nwb.create_device(name="probe-B", description="Synthetic 64-channel probe")
device_b

probe-B pynwb.device.Device at 0x132228274986240
Fields:
  description: Synthetic 64-channel probe

Run the cell below to display the `nwb` object and click on "devices" to see the all probes that you have just added.


In [ ]:
nwb

root pynwb.file.NWBFile at 0x132228272070144
Fields:
  devices: {
    probe-A <class 'pynwb.device.Device'>,
    probe-B <class 'pynwb.device.Device'>
  }
  file_create_date: [datetime.datetime(2026, 6, 22, 12, 36, 59, 814061, tzinfo=tzlocal())]
  identifier: 5647af99-4649-4609-b562-d2c0962207d1
  session_description: Demo NWB file
  session_start_time: 2026-05-01 10:00:00+00:00
  timestamps_reference_time: 2026-05-01 10:00:00+00:00

**Exercise**: create an `ElectrodeGroup` named `'shank0'`, located in `'V1'`, with a description saying "single shank in V1", attached to `device_a`. Assign it to a variable named `group_a`.

**Hint**: see `nwb.create_electrode_group` in code reference table.

In [ ]:
group_a = nwb.create_electrode_group(
    name="shank0",
    description="single shank in V1",
    location="V1",
    device=device_a,
)
group_a

shank0 pynwb.ecephys.ElectrodeGroup at 0x132228274984880
Fields:
  description: single shank in V1
  device: probe-A pynwb.device.Device at 0x132228270792480
Fields:
  description: Synthetic 32-channel probe

  location: V1

Run the cell below to display the `nwb` object and click on "electrode_groups" to see the group you just added.


In [ ]:
nwb

root pynwb.file.NWBFile at 0x132228272070144
Fields:
  devices: {
    probe-A <class 'pynwb.device.Device'>,
    probe-B <class 'pynwb.device.Device'>
  }
  electrode_groups: {
    shank0 <class 'pynwb.ecephys.ElectrodeGroup'>
  }
  file_create_date: [datetime.datetime(2026, 6, 22, 12, 36, 59, 814061, tzinfo=tzlocal())]
  identifier: 5647af99-4649-4609-b562-d2c0962207d1
  session_description: Demo NWB file
  session_start_time: 2026-05-01 10:00:00+00:00
  timestamps_reference_time: 2026-05-01 10:00:00+00:00

**Exercise**: Run the cell below to add four electrodes to the file's electrode table, all belonging to `group_a`, placed them at `x = y = 0`, `z = 0, 1, 2, 3`, in `'V1'`, with `filtering='none'`.


In [ ]:
for i in range(4):
    nwb.add_electrode(
        x=0.0, y=0.0, z=float(i),
        location="V1", group=group_a, filtering="none",
    )
nwb.electrodes.to_dataframe()

,location,group,group_name,x,y,z,filtering
id,,,,,,,
0,V1,shank0 pynwb.ecephys.ElectrodeGroup at 0x13222...,shank0,0.0,0.0,0.0,none
1,V1,shank0 pynwb.ecephys.ElectrodeGroup at 0x13222...,shank0,0.0,0.0,1.0,none
2,V1,shank0 pynwb.ecephys.ElectrodeGroup at 0x13222...,shank0,0.0,0.0,2.0,none
3,V1,shank0 pynwb.ecephys.ElectrodeGroup at 0x13222...,shank0,0.0,0.0,3.0,none


Run the cell below to display the `nwb` object and click on "electrodes" to see the table of electrodes that was added above.


In [ ]:
nwb

,location,group,group_name,x,y,z,filtering
id,,,,,,,
0,V1,shank0 pynwb.ecephys.ElectrodeGroup at 0x132228274984880\nFields:\n description: single shank in V1\n device: probe-A pynwb.device.Device at 0x132228270792480\nFields:\n description: Synthetic 32-channel probe\n\n location: V1\n,shank0,0.0,0.0,0.0,none
1,V1,shank0 pynwb.ecephys.ElectrodeGroup at 0x132228274984880\nFields:\n description: single shank in V1\n device: probe-A pynwb.device.Device at 0x132228270792480\nFields:\n description: Synthetic 32-channel probe\n\n location: V1\n,shank0,0.0,0.0,1.0,none
2,V1,shank0 pynwb.ecephys.ElectrodeGroup at 0x132228274984880\nFields:\n description: single shank in V1\n device: probe-A pynwb.device.Device at 0x132228270792480\nFields:\n description: Synthetic 32-channel probe\n\n location: V1\n,shank0,0.0,0.0,2.0,none
3,V1,shank0 pynwb.ecephys.ElectrodeGroup at 0x132228274984880\nFields:\n description: single shank in V1\n device: probe-A pynwb.device.Device at 0x132228270792480\nFields:\n description: Synthetic 32-channel probe\n\n location: V1\n,shank0,0.0,0.0,3.0,none


**Exercise**: create a second `ElectrodeGroup` and assign it to a variable `group_b`. It should be named `'shank1'`, located in `'A1'`, attached to `device_b`.


In [ ]:
group_b = nwb.create_electrode_group(
    name="shank1",
    description="single shank in A1",
    location="A1",
    device=device_b,
)
group_b

shank1 pynwb.ecephys.ElectrodeGroup at 0x132228274984608
Fields:
  description: single shank in A1
  device: probe-B pynwb.device.Device at 0x132228274986240
Fields:
  description: Synthetic 64-channel probe

  location: A1

**Exercise**: add four electrodes to the file's electrode table, all belonging to `group_b` (not `group_a`), placed them at `x = 0, 1, 2, 3`, `y = z = 0`, in `'A1'`, with `filtering='none'`.

**Hint**: Check how the V1 electrodes were added to the electrode table using `nwb.add_electrode` previously.


In [ ]:
for i in range(4):
    nwb.add_electrode(
        x=float(i), y=0.0, z=0.0,
        location="A1", group=group_b, filtering="none",
    )
nwb.electrodes.to_dataframe()

,location,group,group_name,x,y,z,filtering
id,,,,,,,
0,V1,shank0 pynwb.ecephys.ElectrodeGroup at 0x13222...,shank0,0.0,0.0,0.0,none
1,V1,shank0 pynwb.ecephys.ElectrodeGroup at 0x13222...,shank0,0.0,0.0,1.0,none
2,V1,shank0 pynwb.ecephys.ElectrodeGroup at 0x13222...,shank0,0.0,0.0,2.0,none
3,V1,shank0 pynwb.ecephys.ElectrodeGroup at 0x13222...,shank0,0.0,0.0,3.0,none
4,A1,shank1 pynwb.ecephys.ElectrodeGroup at 0x13222...,shank1,0.0,0.0,0.0,none
5,A1,shank1 pynwb.ecephys.ElectrodeGroup at 0x13222...,shank1,1.0,0.0,0.0,none
6,A1,shank1 pynwb.ecephys.ElectrodeGroup at 0x13222...,shank1,2.0,0.0,0.0,none
7,A1,shank1 pynwb.ecephys.ElectrodeGroup at 0x13222...,shank1,3.0,0.0,0.0,none


**Exercise**: Build an `electrode_table_region` referencing all 8 electrodes you just added.

In [ ]:
region = nwb.create_electrode_table_region(
    region=np.arange(8).tolist(), description="all 8 electrodes",
)
region

electrodes hdmf.common.table.DynamicTableRegion at 0x132228272070448
    Target table: electrodes pynwb.ecephys.ElectrodesTable at 0x132228272021712

## Add Acquisition and Processed Data

### Background

NWB keeps a clear separation between data as it came off the instrument and data you computed from it.

- **Raw data** is stored exactly as acquired, together with a reference back to the electrodes it came from, so the original recording is always preserved untouched.

- **Processed data** is anything derived from the raw signal — filtered LFP, spike-sorting output, behavioural variables such as running speed. These are grouped by topic, with one module per kind of data (e.g. one for electrophysiology-derived signals, one for behaviour). This keeps the file organised and makes clear what is a primary recording and what is an interpretation of it.

In this section you store the raw recordings for each region, then add the processed data (LFP and a behavioral signal) into their respective modules.

### Exercises

| Code | Description |
| --- | --- |
| `ElectricalSeries(name=, data=, electrodes=, rate=, starting_time=)` | Construct an extracellular voltage series |
| `nwb.add_acquisition(es)` | Attach a raw acquisition object to the file |
| `nwb.create_processing_module(name=, description=)` | Create a processing module |
| `TimeSeries(name=, data=, unit=, rate=, starting_time=)` | Generic time series |
| `BehavioralTimeSeries(time_series=, name=)` | Behaviour data interface |
| `LFP(electrical_series=)` | LFP data interface |
| `nwb.processing[name].add(obj)` | Add a data interface to a module |
| `with NWBHDF5IO("my_session.nwb", "w") as io:`<br>&nbsp;&nbsp;&nbsp;&nbsp;`io.write(nwb)`| Write to file |  
| `with NWBHDF5IO("my_session.nwb", "r") as io:`<br>&nbsp;&nbsp;&nbsp;&nbsp;`rnwb = io.read()` | Read the file |

Run the cell below to create a new nwb object and a region of electrodes used in this section.

In [ ]:
nwb = NWBFile(
    session_description="Demo NWB file",
    identifier=str(uuid4()),
    session_start_time=datetime(2026, 5, 1, 10, 0, tzinfo=timezone.utc),
)

region_V1 = utils.create_electrode_table_region(nwb, probe_name='probe-A', shank_nr = 0, area = 'V1', num_electrodes=4)
region_A1 = utils.create_electrode_table_region(nwb, probe_name='probe-B', shank_nr = 1, area = 'A1', num_electrodes=4)

In [ ]:
nwb

,location,group,group_name,x,y,z,filtering
id,,,,,,,
0,V1,shank0 pynwb.ecephys.ElectrodeGroup at 0x132228272015312\nFields:\n description: single shank in V1\n device: probe-A pynwb.device.Device at 0x132228272014352\nFields:\n description: Synthetic 64-channel probe\n\n location: V1\n,shank0,0.0,0.0,0.0,none
1,V1,shank0 pynwb.ecephys.ElectrodeGroup at 0x132228272015312\nFields:\n description: single shank in V1\n device: probe-A pynwb.device.Device at 0x132228272014352\nFields:\n description: Synthetic 64-channel probe\n\n location: V1\n,shank0,0.0,0.0,1.0,none
2,V1,shank0 pynwb.ecephys.ElectrodeGroup at 0x132228272015312\nFields:\n description: single shank in V1\n device: probe-A pynwb.device.Device at 0x132228272014352\nFields:\n description: Synthetic 64-channel probe\n\n location: V1\n,shank0,0.0,0.0,2.0,none
3,V1,shank0 pynwb.ecephys.ElectrodeGroup at 0x132228272015312\nFields:\n description: single shank in V1\n device: probe-A pynwb.device.Device at 0x132228272014352\nFields:\n description: Synthetic 64-channel probe\n\n location: V1\n,shank0,0.0,0.0,3.0,none


**Example**: for the random data below, construct an `ElectricalSeries` with the name `raw_recording_V1`, electrodes set to `region_V1`, a sampling rate of `30000.0` Hz, and `starting_time=0.0`. Add it to `nwb.acquisition`.

In [ ]:
data = np.random.normal(size = (60000, 4))
data.shape, data.dtype

es = ElectricalSeries(
    name="raw_recording_V1",
    data=data,
    electrodes=region_V1,
    rate=30000.0,
    starting_time=0.0,
)
nwb.add_acquisition(es)

**Exercise**: Run the cell below to display the `nwb` object and click on acquisition to see the data that was added.

In [ ]:
nwb

root pynwb.file.NWBFile at 0x132228272014672
Fields:
  acquisition: {
    raw_recording_V1 <class 'pynwb.ecephys.ElectricalSeries'>
  }
  devices: {
    probe-A <class 'pynwb.device.Device'>,
    probe-B <class 'pynwb.device.Device'>
  }
  electrode_groups: {
    shank0 <class 'pynwb.ecephys.ElectrodeGroup'>,
    shank1 <class 'pynwb.ecephys.ElectrodeGroup'>
  }
  electrodes: electrodes <class 'pynwb.ecephys.ElectrodesTable'>
  file_create_date: [datetime.datetime(2026, 6, 22, 12, 24, 45, 390620, tzinfo=tzlocal())]
  identifier: 6335a05c-0f39-4bfa-b55f-4f3776eff000
  session_description: Demo NWB file
  session_start_time: 2026-05-01 10:00:00+00:00
  timestamps_reference_time: 2026-05-01 10:00:00+00:00

**Exercise**: for the random data below, construct an `ElectricalSeries` with the name `raw_recording_A1`, `region_A1` for electrodes, a sampling rate of `30000.0` Hz, and `starting_time=0.0`. Add it to `nwb.acquisition`.

In [ ]:
data = np.random.normal(size = (60000, 4))

es = ElectricalSeries(
    name="raw_recording_A1",
    data=data,
    electrodes=region_A1,
    rate=30000.0,
    starting_time=0.0,
)
nwb.add_acquisition(es)

Run the cell below to display the `nwb` object and click on "processing" to see the added behavior module.

In [ ]:
nwb

root pynwb.file.NWBFile at 0x132228272014672
Fields:
  acquisition: {
    raw_recording_A1 <class 'pynwb.ecephys.ElectricalSeries'>,
    raw_recording_V1 <class 'pynwb.ecephys.ElectricalSeries'>
  }
  devices: {
    probe-A <class 'pynwb.device.Device'>,
    probe-B <class 'pynwb.device.Device'>
  }
  electrode_groups: {
    shank0 <class 'pynwb.ecephys.ElectrodeGroup'>,
    shank1 <class 'pynwb.ecephys.ElectrodeGroup'>
  }
  electrodes: electrodes <class 'pynwb.ecephys.ElectrodesTable'>
  file_create_date: [datetime.datetime(2026, 6, 22, 12, 24, 45, 390620, tzinfo=tzlocal())]
  identifier: 6335a05c-0f39-4bfa-b55f-4f3776eff000
  session_description: Demo NWB file
  session_start_time: 2026-05-01 10:00:00+00:00
  timestamps_reference_time: 2026-05-01 10:00:00+00:00

**Example**: create a processing module named `'ecephys'` for electrophysiology-derived data.


In [ ]:
nwb.create_processing_module(
    name="ecephysV1", description="Derived ecephys V1",
)

ecephysV1 pynwb.base.ProcessingModule at 0x132228272016592
Fields:
  description: Derived ecephys V1

Run the cell below to display the `nwb` object and click on "processing" to see the added ecephys module.

In [ ]:
nwb

root pynwb.file.NWBFile at 0x132228272014672
Fields:
  acquisition: {
    raw_recording_A1 <class 'pynwb.ecephys.ElectricalSeries'>,
    raw_recording_V1 <class 'pynwb.ecephys.ElectricalSeries'>
  }
  devices: {
    probe-A <class 'pynwb.device.Device'>,
    probe-B <class 'pynwb.device.Device'>
  }
  electrode_groups: {
    shank0 <class 'pynwb.ecephys.ElectrodeGroup'>,
    shank1 <class 'pynwb.ecephys.ElectrodeGroup'>
  }
  electrodes: electrodes <class 'pynwb.ecephys.ElectrodesTable'>
  file_create_date: [datetime.datetime(2026, 6, 22, 12, 24, 45, 390620, tzinfo=tzlocal())]
  identifier: 6335a05c-0f39-4bfa-b55f-4f3776eff000
  processing: {
    ecephysV1 <class 'pynwb.base.ProcessingModule'>
  }
  session_description: Demo NWB file
  session_start_time: 2026-05-01 10:00:00+00:00
  timestamps_reference_time: 2026-05-01 10:00:00+00:00

Run the cell below to create an ElectricalSeries object for LFP from the four channels in V1.

In [ ]:
es_lfp = ElectricalSeries(
    name="lfp", data=np.random.normal(size = (2000, 4)), electrodes=region_V1,
    rate=1000.0, starting_time=0.0,
)

**Exercise**: add the LFP Electrical series from the **V1** electrodes to the `ecephysV1` module.


In [ ]:
nwb.processing['ecephysV1'].add(LFP(electrical_series=es_lfp))

LFP pynwb.ecephys.LFP at 0x132228272013392
Fields:
  electrical_series: {
    lfp <class 'pynwb.ecephys.ElectricalSeries'>
  }

Run the cell below to display the `nwb` object and click on "processing" to see the added lfp time series.

In [ ]:
nwb

root pynwb.file.NWBFile at 0x132228272014672
Fields:
  acquisition: {
    raw_recording_A1 <class 'pynwb.ecephys.ElectricalSeries'>,
    raw_recording_V1 <class 'pynwb.ecephys.ElectricalSeries'>
  }
  devices: {
    probe-A <class 'pynwb.device.Device'>,
    probe-B <class 'pynwb.device.Device'>
  }
  electrode_groups: {
    shank0 <class 'pynwb.ecephys.ElectrodeGroup'>,
    shank1 <class 'pynwb.ecephys.ElectrodeGroup'>
  }
  electrodes: electrodes <class 'pynwb.ecephys.ElectrodesTable'>
  file_create_date: [datetime.datetime(2026, 6, 22, 12, 24, 45, 390620, tzinfo=tzlocal())]
  identifier: 6335a05c-0f39-4bfa-b55f-4f3776eff000
  processing: {
    ecephysV1 <class 'pynwb.base.ProcessingModule'>
  }
  session_description: Demo NWB file
  session_start_time: 2026-05-01 10:00:00+00:00
  timestamps_reference_time: 2026-05-01 10:00:00+00:00

**Exercise**: create a processing module named `'ecephysA1'` for electrophysiology-derived data. **Hint**: See example above where `ecephysV1` was added.


In [ ]:
nwb.create_processing_module(
    name="ecephysA1", description="Derived ecephys A1",
)

ecephysA1 pynwb.base.ProcessingModule at 0x132228272013712
Fields:
  description: Derived ecephys A1

Run the cell below to create an ElectricalSeries object for LFP from the four channels in A1.

In [ ]:
es_lfp = ElectricalSeries(
    name="lfp", data=np.random.normal(size = (2000, 4)), electrodes=region_A1,
    rate=1000.0, starting_time=0.0,
)

**Exercise**: add the LFP Electrical series from the **A1** electrodes to the `ecephys` module.


In [ ]:
nwb.processing['ecephysA1'].add(LFP(electrical_series=es_lfp))

LFP pynwb.ecephys.LFP at 0x132228272066800
Fields:
  electrical_series: {
    lfp <class 'pynwb.ecephys.ElectricalSeries'>
  }

Run the cell below to display the `nwb` object and click on "processing" to see the added lfp time series.

In [ ]:
nwb

root pynwb.file.NWBFile at 0x132228272014672
Fields:
  acquisition: {
    raw_recording_A1 <class 'pynwb.ecephys.ElectricalSeries'>,
    raw_recording_V1 <class 'pynwb.ecephys.ElectricalSeries'>
  }
  devices: {
    probe-A <class 'pynwb.device.Device'>,
    probe-B <class 'pynwb.device.Device'>
  }
  electrode_groups: {
    shank0 <class 'pynwb.ecephys.ElectrodeGroup'>,
    shank1 <class 'pynwb.ecephys.ElectrodeGroup'>
  }
  electrodes: electrodes <class 'pynwb.ecephys.ElectrodesTable'>
  file_create_date: [datetime.datetime(2026, 6, 22, 12, 24, 45, 390620, tzinfo=tzlocal())]
  identifier: 6335a05c-0f39-4bfa-b55f-4f3776eff000
  processing: {
    ecephysA1 <class 'pynwb.base.ProcessingModule'>,
    ecephysV1 <class 'pynwb.base.ProcessingModule'>
  }
  session_description: Demo NWB file
  session_start_time: 2026-05-01 10:00:00+00:00
  timestamps_reference_time: 2026-05-01 10:00:00+00:00

**Exercise**: create a `'behavior'` processing module. **Hint**: Look at how the `ecephysV1` processing module was created.


In [ ]:
nwb.create_processing_module(
    name="behavior", description="Behavioural data",
)

behavior pynwb.base.ProcessingModule at 0x132228272065280
Fields:
  description: Behavioural data

Run the cell below to build a `TimeSeries` of synthetic running speed (200 samples at 10 Hz, in `'cm/s'`) and put it in a `BehavioralTimeSeries` with the name field `name = 'behavior-time-series'`.

In [ ]:
running_speed = TimeSeries(
    name="running_speed",
    data=np.random.uniform(0, 30, size=200),
    unit="cm/s",
    rate=10.0,
    starting_time=0.0,
)
bts = BehavioralTimeSeries(time_series=running_speed, name="behavior-time-series")

**Exercise**: add the behavioral time series containing the running speed the `behavior` module.


In [ ]:
nwb.processing["behavior"].add(bts)

Data type,float64
Shape,"(200,)"
Array size,1.56 KiB


Run the cell below to display the `nwb` object and click on "processing" to see the added behavior time series.

In [ ]:
nwb

root pynwb.file.NWBFile at 0x132228272014672
Fields:
  acquisition: {
    raw_recording_A1 <class 'pynwb.ecephys.ElectricalSeries'>,
    raw_recording_V1 <class 'pynwb.ecephys.ElectricalSeries'>
  }
  devices: {
    probe-A <class 'pynwb.device.Device'>,
    probe-B <class 'pynwb.device.Device'>
  }
  electrode_groups: {
    shank0 <class 'pynwb.ecephys.ElectrodeGroup'>,
    shank1 <class 'pynwb.ecephys.ElectrodeGroup'>
  }
  electrodes: electrodes <class 'pynwb.ecephys.ElectrodesTable'>
  file_create_date: [datetime.datetime(2026, 6, 22, 12, 24, 45, 390620, tzinfo=tzlocal())]
  identifier: 6335a05c-0f39-4bfa-b55f-4f3776eff000
  processing: {
    behavior <class 'pynwb.base.ProcessingModule'>,
    ecephysA1 <class 'pynwb.base.ProcessingModule'>,
    ecephysV1 <class 'pynwb.base.ProcessingModule'>
  }
  session_description: Demo NWB file
  session_start_time: 2026-05-01 10:00:00+00:00
  timestamps_reference_time: 2026-05-01 10:00:00+00:00

**Exercise**: write the file to `my_session.nwb` using `NWBHDF5IO`.


In [ ]:
with NWBHDF5IO("my_session.nwb", "w") as io:
    io.write(nwb)
print("wrote my_session.nwb")

wrote my_session.nwb


## Writing Tabular Data with DynamicTables: Trials and Spike Units

### Background

Beyond the recorded signals, an NWB file also captures the structure of the experiment and the results of analysing it. The **trials** table divides the session into labeled time intervals, each with a start and stop, plus any extra information such as the stimulus shown or the choice made, turning a continuous recording into the events most analyses are built around. The **units** table holds the spiking activity of individual neurons found by spike sorting, essentially a list of firing times that can be aligned to those trials.

In this section you build a small trial table with a stimulus label and add a couple of sorted units with synthetic spike times.

### Exercises

| Code | Description |
| --- | --- |
| `nwb.add_trial_column(name=, description=)` | Define a new trial column (before adding rows) |
| `nwb.add_trial(start_time=, stop_time=, ...)` | Add a trial row |
| `nwb.trials.to_dataframe()` | Inspect the trial table |
| `nwb.add_unit(spike_times=...)` | Add a sorted unit |
| `nwb.units.to_dataframe()` | Inspect the units table |
| `len(nwb.units)` | Number of units |


Run the cell below to create a new nwb object used in this section.

In [ ]:
nwb = NWBFile(
    session_description="Demo NWB file",
    identifier=str(uuid4()),
    session_start_time=datetime(2026, 5, 1, 10, 0, tzinfo=timezone.utc),
)

**Example**: add a `stim_type` column to the trial table and add one trial.


In [ ]:
nwb.add_trial_column(name="stim_type", description="Stimulus identifier")
nwb.add_trial(start_time=0.0, stop_time=0.3, stim_type="A")

Run the cell below to display the `nwb` object and click on "trials" to see the trial table that was just added.

In [ ]:
nwb

,start_time,stop_time,stim_type
id,,,
0,0.0,0.3,A


**Exercise**: add a second trial with `start_time=0.5`, `stop_time=0.8`, `stim_type='B'`.


In [ ]:
nwb.add_trial(start_time=0.5, stop_time=0.8, stim_type="B")

**Exercise**: add a third trial with `start_time=1.0`, `stop_time=1.3`, `stim_type='A'`.


In [ ]:
nwb.add_trial(start_time=1.0, stop_time=1.3, stim_type="A")

Run the cell below to display the `nwb` object and click on "trials" to see that two new trials were added to the trial table.

In [ ]:
nwb

,start_time,stop_time,stim_type
id,,,
0,0.0,0.3,A
1,0.5,0.8,B
2,1.0,1.3,A


**Exercise**: add a unit to `nwb.units` with synthetic spike times. Use `np.sort(np.random.uniform(0, 2, size=4))` to create a unit with 4 random spikes in a two-second window sorted from the earliest to the latest.


In [ ]:
nwb.add_unit(spike_times=np.sort(np.random.uniform(0, 2, size=4)))

Run the cell below to display the `nwb` object and click on "units" to see the added unit.

In [ ]:
nwb

root pynwb.file.NWBFile at 0x123994756926032
Fields:
  file_create_date: [datetime.datetime(2026, 6, 22, 11, 35, 0, 665502, tzinfo=tzlocal())]
  identifier: 265d550a-9577-49dd-b642-d0518a2bb71c
  session_description: Demo NWB file
  session_start_time: 2026-05-01 10:00:00+00:00
  timestamps_reference_time: 2026-05-01 10:00:00+00:00
  trials: trials <class 'pynwb.epoch.TimeIntervals'>
  units: units <class 'pynwb.misc.Units'>

**Exercise**: add another unit to `nwb.units` with synthetic spike times. Use `np.sort(np.random.uniform(0, 2, size=8))` to create a unit with 8 random spikes within a two-second window sorted from the earliest to the latest.


In [ ]:
nwb.add_unit(spike_times=np.sort(np.random.uniform(0, 2, size=8)))

Run the cell below to display the `nwb` object and click on "units" to see all added units.

In [ ]:
nwb

root pynwb.file.NWBFile at 0x123994756926032
Fields:
  file_create_date: [datetime.datetime(2026, 6, 22, 11, 35, 0, 665502, tzinfo=tzlocal())]
  identifier: 265d550a-9577-49dd-b642-d0518a2bb71c
  session_description: Demo NWB file
  session_start_time: 2026-05-01 10:00:00+00:00
  timestamps_reference_time: 2026-05-01 10:00:00+00:00
  trials: trials <class 'pynwb.epoch.TimeIntervals'>
  units: units <class 'pynwb.misc.Units'>

**Exercise**: Add the timestamps of the 5 spike trains in `all_spike_times` generated below to the nwb file.

In [ ]:
all_spike_times = np.sort(np.random.uniform(0,2, size = (5, 8)), axis = 1)

In [ ]:
for i in range(5):
    nwb.add_unit(spike_times = all_spike_times[i])

Run the cell below to display the `nwb` object and click on "units" to see all added units.

In [ ]:
nwb

root pynwb.file.NWBFile at 0x123994756926032
Fields:
  file_create_date: [datetime.datetime(2026, 6, 22, 11, 35, 0, 665502, tzinfo=tzlocal())]
  identifier: 265d550a-9577-49dd-b642-d0518a2bb71c
  session_description: Demo NWB file
  session_start_time: 2026-05-01 10:00:00+00:00
  timestamps_reference_time: 2026-05-01 10:00:00+00:00
  trials: trials <class 'pynwb.epoch.TimeIntervals'>
  units: units <class 'pynwb.misc.Units'>